# CBD Adapter — MuJoCo

**`adapter/` role of the [Common Behavior Data](https://github.com/Koichi3333/common-behavior-data) project.**
Target: **MuJoCo** (`humanoid.xml` + `motion.npz`).

This notebook reads the canonical dataset written by
`cbd_generator_video_to_cbd.ipynb` and projects it onto a MuJoCo humanoid: a
T-pose MJCF skeleton, a qpos trajectory, an object proxy track, a local replay
script and a rendered video. It never runs MediaPipe and never reads another
adapter's output.

```text
cbd_dataset.zip ─▶ [ THIS NOTEBOOK ] ─▶ output/02_mujoco/humanoid.xml
 (canonical CBD)     Y-up ➜ Z-up                          motion.npz
                                                          object_trajectory.csv
                                                          replay_mujoco.py
                                                          mujoco_simulation.mp4
                                                          adapter_report.json
```

## Adapter rules honoured here

1. **Reads canonical data** — `timeline/frames.jsonl` and `manifest.json` only
2. **Converts at its boundary** — canonical Y-up right-handed becomes MuJoCo
   Z-up in `[A4]`, by conjugating every bone quaternion with a +90° rotation
   about X. Nowhere else in the notebook are the two frames mixed.
3. **Model and motion stay separate** — `humanoid.xml` and `motion.npz` are
   two files, and the qpos layout is read back from the compiled model rather
   than assumed
4. **Does not promote candidates** — the phase band says "Grasp Candidate"
5. **Reports what it could not represent** in `adapter_report.json`
   (face, blendshapes, per-joint finger segments, …)
6. **Preserves provenance** — `object_trajectory.csv` keeps the
   `position_source` of every object position

Playback mode is **kinematic replay** (`data.qpos[:] = recorded_qpos[frame]`
plus `mj_forward`). It is not a claim about physically correct contact or
forces.

## Input

Any one of these, tried in order — no editing needed:

1. the dataset already unpacked in this runtime (the generator just ran here)
2. `/content/cbd_dataset.zip` (`colab upload`)
3. the Colab upload widget (UI only)

## How to run — colab CLI

```bash
colab exec -s cbd -f cbd_adapter_mujoco.ipynb --timeout 1800
colab download -s cbd \
  /content/human_behavior_demo_2_0/mujoco_adapter_output.zip ./mujoco_adapter_output.zip
```

A T4 GPU renders with EGL; on CPU the adapter falls back to OSMesa software
rendering, which is slower but completes. Rendering runs in a subprocess so a
GL failure cannot take the kernel down with it.


In [ ]:
# @title [A1] Environment setup + rendering backend
# =====================================================================
# MuJoCo adapter 1/6. Run once per runtime.
#   GPU runtime -> egl    (we generate the EGL ICD config file ourselves)
#   CPU runtime -> osmesa (software rendering)
# =====================================================================
import json
import math
import os
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

print("--- Installing packages (takes 1-2 minutes) ---")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "mujoco", "imageio-ffmpeg"],
    check=False,
)

gpu_query = subprocess.run(
    "nvidia-smi --query-gpu=name --format=csv,noheader",
    shell=True, capture_output=True, text=True,
)
HAS_GPU = gpu_query.returncode == 0
GPU_NAME = gpu_query.stdout.strip().splitlines()[0] if HAS_GPU and gpu_query.stdout.strip() else None

if HAS_GPU:
    # Colab ships without an EGL ICD config. Without it MuJoCo aborts the
    # whole process with mju_error and the kernel restarts -- writing this
    # tiny JSON file is the fix.
    NVIDIA_ICD_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(NVIDIA_ICD_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_PATH), exist_ok=True)
        with open(NVIDIA_ICD_PATH, "w") as icd_file:
            icd_file.write(
                '{"file_format_version":"1.0.0",'
                '"ICD":{"library_path":"libEGL_nvidia.so.0"}}'
            )
        print("Created EGL ICD config:", NVIDIA_ICD_PATH)
    os.environ["MUJOCO_GL"] = "egl"
    print(f"GPU: {GPU_NAME} / renderer: egl")
else:
    subprocess.run("apt-get -qq install -y libosmesa6-dev > /dev/null",
                   shell=True, check=False)
    os.environ["MUJOCO_GL"] = "osmesa"
    os.environ["PYOPENGL_PLATFORM"] = "osmesa"
    print("No GPU / renderer: osmesa (CPU, roughly 5-10x slower)")
    print("Tip: Runtime > Change runtime type > T4 GPU")

import cv2
import mujoco

print("mujoco:", mujoco.__version__)

# ---------------------------------------------------------------
# Run mode: Colab UI or headless (colab CLI / Run all / papermill)
# ---------------------------------------------------------------
# The two entry points differ in exactly one way that matters here: whether
# the kernel accepts stdin. `colab exec` calls execute_code() without
# allow_stdin, so an upload widget would hang until the run times out, and an
# embedded base64 video player would dump megabytes into the log. Detect the
# mode once, then degrade gracefully instead of blocking.
def _stdin_available():
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or not hasattr(shell, "kernel"):
            return False
        return bool(getattr(shell.kernel, "_allow_stdin", False))
    except Exception:  # noqa: BLE001
        return False


IS_COLAB_UI = _stdin_available()
SHOW_MEDIA = IS_COLAB_UI      # embed videos / images only in the browser UI
RUN_MODE = "colab-ui" if IS_COLAB_UI else "headless (colab CLI / Run all)"
print(f"Run mode: {RUN_MODE}")
# ------------------------- Shared media utilities -------------------------
def encode_h264(source_path, destination_path, fps):
    """Transcode to H.264 with ffmpeg so the video plays in the browser."""
    command = ["ffmpeg", "-y", "-loglevel", "error", "-r", str(fps),
               "-i", str(source_path), "-c:v", "libx264",
               "-pix_fmt", "yuv420p", "-movflags", "+faststart",
               str(destination_path)]
    result = subprocess.run(command, capture_output=True)
    if result.returncode != 0 or not Path(destination_path).exists():
        shutil.copy(source_path, destination_path)
    return str(destination_path)


def show_video(video_path, width=520):
    """Embed a video player -- Colab UI only.

    In a headless run (colab CLI) the base64 payload would be written to the
    execution log, so print the path instead.
    """
    if not SHOW_MEDIA:
        size_mb = Path(video_path).stat().st_size / 1024 / 1024
        print(f"[preview skipped in headless mode] {video_path} ({size_mb:.1f} MB)")
        return
    import base64
    from IPython.display import HTML, display
    data = Path(video_path).read_bytes()
    if len(data) > 40 * 1024 * 1024:
        print(f"Video too large to embed, skipping preview: {video_path} "
              f"({len(data) / 1024 / 1024:.1f} MB)")
        return
    encoded = base64.b64encode(data).decode()
    display(HTML(f'<video width="{width}" controls loop playsinline '
                 f'src="data:video/mp4;base64,{encoded}"></video>'))


def make_video_writer(path, fps, size):
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"),
                             fps, size)
    if not writer.isOpened():
        raise RuntimeError(f"Could not create the output video: {path}")
    return writer


def even(value):
    return int(value) - int(value) % 2


In [ ]:
# @title [A2] Load the Common Behavior Data
# ---------------------------------------------------------------
# Locate the Common Behavior Data
# ---------------------------------------------------------------
# Resolution order, so the same cell works from either entry point:
#   1. an already-unpacked dataset in this runtime (the generator just ran)
#   2. cbd_dataset.zip sitting on the VM (`colab upload`)
#   3. the upload widget -- Colab UI only
#   4. otherwise: stop with the exact command needed, instead of hanging
CBD_DIR_OVERRIDE = ""      # optional: a directory containing 04_behavior_dataset/

PROJECT_DIR = Path("/content/human_behavior_demo_2_0")
OUTPUT_DIR = PROJECT_DIR / "output"
WORK_DIR = PROJECT_DIR / "_work"
ZIP_CANDIDATES = ["/content/cbd_dataset.zip", "cbd_dataset.zip",
                  str(PROJECT_DIR / "cbd_dataset.zip")]


def _find_dataset(root):
    """Return the 04_behavior_dataset directory below root, if any."""
    root = Path(root)
    direct = root / "output/04_behavior_dataset/manifest.json"
    if direct.exists():
        return direct.parent
    if (root / "manifest.json").exists() and root.name == "04_behavior_dataset":
        return root
    hit = next(iter(sorted(root.glob("**/04_behavior_dataset/manifest.json"))), None)
    return hit.parent if hit else None


def _unpack(zip_path):
    print(f"Unpacking {zip_path} -> {PROJECT_DIR}")
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(PROJECT_DIR)


DATASET_DIR = _find_dataset(CBD_DIR_OVERRIDE) if CBD_DIR_OVERRIDE else None
if DATASET_DIR is None:
    DATASET_DIR = _find_dataset(PROJECT_DIR)
if DATASET_DIR is None:
    _zip = next((p for p in ZIP_CANDIDATES if Path(p).exists()), None)
    if _zip is None and IS_COLAB_UI:
        print("No Common Behavior Data on this VM. Upload cbd_dataset.zip "
              "(produced by cbd_generator_video_to_cbd.ipynb).")
        from google.colab import files
        _uploaded = files.upload()
        _zip = "/content/" + next(iter(_uploaded))
    if _zip is None:
        raise RuntimeError(
            "No Common Behavior Data on this VM and this is a headless run, "
            "so no upload widget can be opened.\n"
            "Run the generator on this session first, or upload its output:\n"
            "    colab upload -s <session> ./cbd_dataset.zip "
            "/content/cbd_dataset.zip")
    _unpack(_zip)
    DATASET_DIR = _find_dataset(PROJECT_DIR)
if DATASET_DIR is None:
    raise RuntimeError("cbd_dataset.zip does not contain 04_behavior_dataset/")

OUTPUT_DIR = DATASET_DIR.parent
# Normal layout: <project>/output/04_behavior_dataset. A flat bundle (the
# dataset directly under the extract root) is accepted too -- then the extract
# root doubles as the project root, so nothing is written outside it.
PROJECT_DIR = OUTPUT_DIR.parent if OUTPUT_DIR.name == "output" else OUTPUT_DIR
WORK_DIR = PROJECT_DIR / "_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_VIDEO = PROJECT_DIR / "source/source_video.mp4"
if not SOURCE_VIDEO.exists():
    SOURCE_VIDEO = next(iter(sorted(PROJECT_DIR.glob("**/source_video.mp4"))),
                        SOURCE_VIDEO)

print("Common Behavior Data:", DATASET_DIR)

# ---------------------------------------------------------------
# Read the canonical timeline
# ---------------------------------------------------------------
# Everything below comes from files. No MediaPipe, no re-computation: that is
# the whole point of the adapter boundary.
MANIFEST = json.loads((DATASET_DIR / "manifest.json").read_text(encoding="utf-8"))
SUMMARY = json.loads(
    (DATASET_DIR / "behavior_summary.json").read_text(encoding="utf-8"))
GENERATOR_CONFIG = {}
if (PROJECT_DIR / "config.json").exists():
    GENERATOR_CONFIG = json.loads(
        (PROJECT_DIR / "config.json").read_text(encoding="utf-8"))

FRAMES = [json.loads(line) for line
          in (DATASET_DIR / "timeline/frames.jsonl").read_text(
              encoding="utf-8").splitlines() if line.strip()]
NUM_FRAMES = len(FRAMES)
if NUM_FRAMES == 0:
    raise RuntimeError("timeline/frames.jsonl is empty")
FPS = float(MANIFEST["fps"])
DT = 1.0 / FPS
TIMESTAMPS = np.array([f["timestamp_sec"] for f in FRAMES], dtype=np.float32)
SOURCE_FRAME_INDEX = [f["source_frame_index"] for f in FRAMES]

BONE_ORDER = (MANIFEST.get("bone_order")
              or list(FRAMES[0]["human"]["bone_rotations_xyzw"]))
FINGER_ORDER = MANIFEST.get("finger_order",
                            ["thumb", "index", "middle", "ring", "little"])

# Bone rotations: stored xyzw in the file, used wxyz by the quaternion helpers
BONE_ROT = {}
for bone in BONE_ORDER:
    xyzw = np.array([f["human"]["bone_rotations_xyzw"][bone] for f in FRAMES],
                    dtype=np.float64)
    BONE_ROT[bone] = np.column_stack([xyzw[:, 3], xyzw[:, 0], xyzw[:, 1],
                                      xyzw[:, 2]])
HIPS_POS = np.array([f["human"]["hips_position"] for f in FRAMES],
                    dtype=np.float64)

# Finger curls [rad]. The timeline stores null on frames where the hand was
# not detected; fill from the nearest valid frame so a renderer does not snap
# the hand open, and record how many frames that affected.
ADAPTER_NOTES = []       # what this adapter could not represent, or had to fill
CURLS, HAND_PRESENT, CURLS_FILLED = {}, {}, {}
for _side in ["left", "right"]:
    raw = [f["human"]["finger_curls_rad"].get(_side) for f in FRAMES]
    present = np.array([r is not None for r in raw])
    values = np.zeros((NUM_FRAMES, len(FINGER_ORDER)))
    if present.any():
        valid_idx = np.where(present)[0]
        for fi in range(NUM_FRAMES):
            src = fi if present[fi] else valid_idx[np.argmin(np.abs(valid_idx - fi))]
            values[fi] = raw[src]
    CURLS[_side.capitalize()] = values
    HAND_PRESENT[_side.capitalize()] = present
    CURLS_FILLED[_side.capitalize()] = int((~present).sum()) if present.any() else 0
    if present.any() and CURLS_FILLED[_side.capitalize()]:
        ADAPTER_NOTES.append(
            f"finger curls: {CURLS_FILLED[_side.capitalize()]}/{NUM_FRAMES} "
            f"{_side} frames had no hand detection and were filled from the "
            "nearest detected frame")
    if not present.any():
        ADAPTER_NOTES.append(f"finger curls: no {_side} hand was ever detected")

PHASE = [f["phase"] for f in FRAMES]
INTERACTIONS = [f["interactions"] for f in FRAMES]
CAPTIONS = [f.get("caption") for f in FRAMES]

# Primary object. "primary_object" is defined on every frame; datasets written
# before that field existed are read back from the objects[] list instead.
PRIMARY_LABEL = (MANIFEST.get("primary", {}).get("object_label")
                 or (SUMMARY.get("primary_object") or {}).get("label"))
PRIMARY_TRACK_ID = (MANIFEST.get("primary", {}).get("object_track_id")
                    or (SUMMARY.get("primary_object") or {}).get("track_id"))
PRIMARY_HAND = (MANIFEST.get("primary", {}).get("hand")
                or SUMMARY.get("primary_hand") or "right")

OBJ_PROXY, OBJ_SOURCE = None, ["none"] * NUM_FRAMES
if PRIMARY_TRACK_ID:
    positions, sources, missing = [], [], 0
    for f in FRAMES:
        block = f.get("primary_object")
        if block is None:
            block = next((o for o in f["objects"]
                          if o.get("track_id") == PRIMARY_TRACK_ID
                          and "proxy_canonical" in o), None)
        if block and block.get("proxy_canonical"):
            positions.append(block["proxy_canonical"])
            sources.append(block.get("position_source", "unknown"))
        else:
            positions.append(positions[-1] if positions else [0.0, 0.9, 0.3])
            sources.append("carried_forward_by_adapter")
            missing += 1
    OBJ_PROXY = np.array(positions, dtype=np.float64)
    OBJ_SOURCE = sources
    if missing:
        ADAPTER_NOTES.append(
            f"object proxy: {missing}/{NUM_FRAMES} frames had no canonical "
            "position in the dataset and were carried forward by this adapter")

print(f"frames={NUM_FRAMES}  fps={FPS:.2f}  duration={TIMESTAMPS[-1]:.2f}s")
print(f"task={SUMMARY.get('task')}  primary_hand={PRIMARY_HAND}  "
      f"primary_object={PRIMARY_LABEL or 'none'}")
print("events:", SUMMARY.get("events") or "none")
print("captions:", sum(1 for c in CAPTIONS if c), "frames carry a caption")
for note in ADAPTER_NOTES:
    print("  note:", note)


def write_adapter_report(path, adapter, target, reads, unrepresented):
    """Rule 5: say what could not be represented, rather than approximating
    it silently. Rule 6: say where every derived value came from."""
    report = {
        "adapter": adapter,
        "target": target,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "source_dataset": {
            "manifest": str(Path(DATASET_DIR) / "manifest.json"),
            "dataset_version": MANIFEST.get("dataset_version"),
            "frame_count": NUM_FRAMES,
            "fps": FPS,
        },
        "reads": reads,
        "filled_or_interpolated": list(ADAPTER_NOTES),
        "not_represented": unrepresented,
        "promoted_candidates": False,
    }
    Path(path).write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("Wrote:", path)
    return report
# ---------------------------------------------------------------
# Quaternion helpers (wxyz), used for the canonical -> MuJoCo conversion
# ---------------------------------------------------------------
def q_normalize(q):
    return q / (np.linalg.norm(q) + 1e-12)


def q_mul(a, b):
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,
                     aw*bx + ax*bw + ay*bz - az*by,
                     aw*by - ax*bz + ay*bw + az*bx,
                     aw*bz + ax*by - ay*bx + az*bw])


def q_conj(q):
    return np.array([q[0], -q[1], -q[2], -q[3]])


print("bones:", len(BONE_ROT), "| object proxy:",
      "yes" if OBJ_PROXY is not None else "no")


In [ ]:
# @title [A3] Build the humanoid model (MJCF)
# =====================================================================
# MuJoCo adapter 3/6.
# Rule 3 of an adapter: keep model and motion separate where the target
# format allows it. MuJoCo does, so the skeleton is written here and the
# trajectory in [A4]; motion.npz is a projection of the behavior data, never
# a MuJoCo original.
# =====================================================================
MUJOCO_CFG = {
    "render_width": 960,
    "render_height": 720,
    # Rest height of the pelvis in the MJCF. The dataset's hips_position is a
    # translation offset around the origin, so this adapter supplies its own
    # rest height (see manifest.canonical_scale.hips_position_note).
    "rest_hip_height": 0.93,
    "camera_distance": 3.4,
}

MUJOCO_DIR = OUTPUT_DIR / "02_mujoco"
MUJOCO_DIR.mkdir(parents=True, exist_ok=True)
# A T-pose skeleton (arms along +/-X, legs along -Z, person facing -Y).
# Every body is defined with zero rotation, so each ball joint's qpos is
# exactly the canonical bone rotation after a coordinate transform -- no
# retargeting, no IK, no hidden fitting step.
FINGER_Y_LEFT = {"thumb": -0.042, "index": -0.024, "middle": -0.008,
                 "ring": 0.008, "little": 0.024}

MJ_BALL_ORDER = []      # ball joints, in qpos order
MJ_HINGE_ORDER = []     # hinge joints (fingers), in qpos order


def finger_bodies_xml(side):
    prefix = side.lower()
    sign = 1.0 if side == "Left" else -1.0
    axis = "0 1 0" if side == "Left" else "0 -1 0"
    parts = []
    for finger in FINGER_ORDER:
        y = FINGER_Y_LEFT[finger]
        joint_name = f"{prefix}_finger_{finger}"
        MJ_HINGE_ORDER.append(joint_name)
        parts.append(f"""
            <body name="{prefix}_{finger}" pos="{sign*0.088:.3f} {y:.3f} 0">
              <joint name="{joint_name}" type="hinge" axis="{axis}"
                     range="0 2.2" limited="true"/>
              <geom type="capsule" fromto="0 0 0 {sign*0.055:.3f} 0 0"
                    size="0.008" rgba="{FINGER_RGBA}"/>
            </body>""")
    return "".join(parts)


BODY_RGBA = "0.55 0.62 0.72 1"
ACCENT_RGBA = "0.85 0.45 0.25 1"
FINGER_RGBA = "0.80 0.80 0.85 1"


def ball(name):
    MJ_BALL_ORDER.append(name)
    return f'<joint name="{name}" type="ball"/>'


# Object proxy geometry, chosen from the primary object's class.
# Unity uses the same classification, so both engines show the same shape.
CYLINDER_LABELS = {"cup", "bottle", "wine glass", "vase", "bowl"}
SPHERE_LABELS = {"sports ball", "ball", "apple", "orange", "frisbee"}
_obj_label = PRIMARY_LABEL
if _obj_label in CYLINDER_LABELS:
    OBJECT_GEOM_XML = ('<geom type="cylinder" size="0.035 0.07" '
                       'rgba="0.85 0.30 0.30 0.9" contype="0" conaffinity="0"/>')
elif _obj_label in SPHERE_LABELS:
    OBJECT_GEOM_XML = ('<geom type="sphere" size="0.05" '
                       'rgba="0.85 0.30 0.30 0.9" contype="0" conaffinity="0"/>')
else:
    OBJECT_GEOM_XML = ('<geom type="box" size="0.05 0.05 0.09" '
                       'rgba="0.85 0.30 0.30 0.9" contype="0" conaffinity="0"/>')

HUMANOID_XML = f"""<mujoco model="human_behavior_demo_2_0">
  <option gravity="0 0 0"/>
  <visual>
    <global offwidth="{MUJOCO_CFG['render_width']}" offheight="{MUJOCO_CFG['render_height']}"/>
    <quality shadowsize="1024" offsamples="{4 if HAS_GPU else 0}"/>
    <headlight ambient="0.45 0.45 0.45" diffuse="0.7 0.7 0.7"/>
  </visual>
  <asset>
    <texture name="grid" type="2d" builtin="checker" width="256" height="256"
             rgb1="0.20 0.24 0.30" rgb2="0.26 0.30 0.36"/>
    <material name="grid" texture="grid" texrepeat="6 6" reflectance="0.05"/>
  </asset>
  <worldbody>
    <geom name="floor" type="plane" size="4 4 0.05" material="grid"/>
    <light pos="0 -2 3" dir="0 0.5 -1" diffuse="0.8 0.8 0.8"/>
    <body name="pelvis" pos="0 0 {MUJOCO_CFG['rest_hip_height']}">
      <freejoint name="root"/>
      <geom type="box" size="0.11 0.08 0.07" rgba="{BODY_RGBA}"/>
      <body name="spine" pos="0 0 0.10">
        {ball('spine')}
        <geom type="capsule" fromto="0 0 0 0 0 0.10" size="0.07" rgba="{BODY_RGBA}"/>
        <body name="chest" pos="0 0 0.12">
          {ball('chest')}
          <geom type="capsule" fromto="0 0 0 0 0 0.14" size="0.09" rgba="{BODY_RGBA}"/>
          <body name="neck" pos="0 0 0.20">
            {ball('neck')}
            <geom type="capsule" fromto="0 0 0 0 0 0.05" size="0.035" rgba="{BODY_RGBA}"/>
            <body name="head" pos="0 0 0.06">
              {ball('head')}
              <geom type="sphere" pos="0 0 0.08" size="0.09" rgba="{BODY_RGBA}"/>
              <geom type="sphere" pos="0.035 -0.075 0.09" size="0.012" rgba="0.1 0.1 0.1 1"/>
              <geom type="sphere" pos="-0.035 -0.075 0.09" size="0.012" rgba="0.1 0.1 0.1 1"/>
            </body>
          </body>
          <body name="left_shoulder" pos="0.08 0 0.14">
            <geom type="sphere" size="0.045" rgba="{BODY_RGBA}"/>
            <body name="left_upper_arm" pos="0.09 0 0">
              {ball('left_upper_arm')}
              <geom type="capsule" fromto="0 0 0 0.26 0 0" size="0.038" rgba="{BODY_RGBA}"/>
              <body name="left_lower_arm" pos="0.26 0 0">
                {ball('left_lower_arm')}
                <geom type="capsule" fromto="0 0 0 0.25 0 0" size="0.032" rgba="{BODY_RGBA}"/>
                <body name="left_hand" pos="0.25 0 0">
                  {ball('left_hand')}
                  <geom type="box" pos="0.045 0 0" size="0.045 0.032 0.012" rgba="{ACCENT_RGBA}"/>
                  {finger_bodies_xml('Left')}
                </body>
              </body>
            </body>
          </body>
          <body name="right_shoulder" pos="-0.08 0 0.14">
            <geom type="sphere" size="0.045" rgba="{BODY_RGBA}"/>
            <body name="right_upper_arm" pos="-0.09 0 0">
              {ball('right_upper_arm')}
              <geom type="capsule" fromto="0 0 0 -0.26 0 0" size="0.038" rgba="{BODY_RGBA}"/>
              <body name="right_lower_arm" pos="-0.26 0 0">
                {ball('right_lower_arm')}
                <geom type="capsule" fromto="0 0 0 -0.25 0 0" size="0.032" rgba="{BODY_RGBA}"/>
                <body name="right_hand" pos="-0.25 0 0">
                  {ball('right_hand')}
                  <geom type="box" pos="-0.045 0 0" size="0.045 0.032 0.012" rgba="{ACCENT_RGBA}"/>
                  {finger_bodies_xml('Right')}
                </body>
              </body>
            </body>
          </body>
        </body>
      </body>
      <body name="left_upper_leg" pos="0.09 0 -0.05">
        {ball('left_upper_leg')}
        <geom type="capsule" fromto="0 0 0 0 0 -0.42" size="0.055" rgba="{BODY_RGBA}"/>
        <body name="left_lower_leg" pos="0 0 -0.42">
          {ball('left_lower_leg')}
          <geom type="capsule" fromto="0 0 0 0 0 -0.40" size="0.045" rgba="{BODY_RGBA}"/>
          <body name="left_foot" pos="0 0 -0.40">
            {ball('left_foot')}
            <geom type="box" pos="0 -0.05 -0.025" size="0.045 0.10 0.02" rgba="{ACCENT_RGBA}"/>
          </body>
        </body>
      </body>
      <body name="right_upper_leg" pos="-0.09 0 -0.05">
        {ball('right_upper_leg')}
        <geom type="capsule" fromto="0 0 0 0 0 -0.42" size="0.055" rgba="{BODY_RGBA}"/>
        <body name="right_lower_leg" pos="0 0 -0.42">
          {ball('right_lower_leg')}
          <geom type="capsule" fromto="0 0 0 0 0 -0.40" size="0.045" rgba="{BODY_RGBA}"/>
          <body name="right_foot" pos="0 0 -0.40">
            {ball('right_foot')}
            <geom type="box" pos="0 -0.05 -0.025" size="0.045 0.10 0.02" rgba="{ACCENT_RGBA}"/>
          </body>
        </body>
      </body>
    </body>
    <body name="object_proxy" mocap="true" pos="0 -0.3 -5">
      {OBJECT_GEOM_XML}
    </body>
  </worldbody>
</mujoco>
"""

HUMANOID_XML_PATH = MUJOCO_DIR / "humanoid.xml"
HUMANOID_XML_PATH.write_text(HUMANOID_XML, encoding="utf-8")

# Validate the model. Loading alone needs no GL context, so this is safe.
mj_model = mujoco.MjModel.from_xml_string(HUMANOID_XML)
print(f"humanoid.xml OK  nq={mj_model.nq}  ball={len(MJ_BALL_ORDER)}  "
      f"hinge={len(MJ_HINGE_ORDER)}")


In [ ]:
# @title [A4] Canonical rotations -> qpos, motion.npz and the replay script
# =====================================================================
# MuJoCo adapter 4/6. The coordinate conversion happens here, at this
# adapter's boundary -- canonical Y-up right-handed to MuJoCo Z-up.
# =====================================================================
# ------------- Canonical rotations -> qpos -------------
# canonical (Y-up) -> MuJoCo (Z-up): conjugation by a +90 deg rotation about X
Q_C2MJ = np.array([math.cos(math.pi/4), math.sin(math.pi/4), 0.0, 0.0])
Q_C2MJ_INV = q_conj(Q_C2MJ)


def quat_c_to_mj(q):
    return q_normalize(q_mul(Q_C2MJ, q_mul(q, Q_C2MJ_INV)))


def vec_c_to_mj(v):
    return np.array([v[0], -v[2], v[1]])


# The qpos layout follows the XML tree order, so never assume it -- read the
# real joint addresses (jnt_qposadr) back from the compiled model.
NQ = int(mj_model.nq)
JOINT_LAYOUT = []
for joint_id in range(mj_model.njnt):
    JOINT_LAYOUT.append({
        "name": mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_JOINT, joint_id),
        "type": int(mj_model.jnt_type[joint_id]),
        "qpos_adr": int(mj_model.jnt_qposadr[joint_id]),
    })

QPOS = np.zeros((NUM_FRAMES, NQ), dtype=np.float32)
for fi in range(NUM_FRAMES):
    for joint in JOINT_LAYOUT:
        adr = joint["qpos_adr"]
        if joint["type"] == int(mujoco.mjtJoint.mjJNT_FREE):
            QPOS[fi, adr:adr + 3] = (
                vec_c_to_mj(HIPS_POS[fi])
                + np.array([0.0, 0.0, MUJOCO_CFG["rest_hip_height"]]))
            QPOS[fi, adr + 3:adr + 7] = quat_c_to_mj(BONE_ROT["hips"][fi])
        elif joint["type"] == int(mujoco.mjtJoint.mjJNT_BALL):
            QPOS[fi, adr:adr + 4] = quat_c_to_mj(BONE_ROT[joint["name"]][fi])
        else:  # hinge joints are the fingers
            side = "Left" if joint["name"].startswith("left") else "Right"
            finger = joint["name"].split("_")[-1]
            curl = float(CURLS[side][fi, FINGER_ORDER.index(finger)])
            QPOS[fi, adr] = float(np.clip(curl, 0.0, 2.2))

# Object proxy, converted into MuJoCo coordinates
OBJ_MJ = None
if OBJ_PROXY is not None:
    OBJ_MJ = np.stack([vec_c_to_mj(p) for p in OBJ_PROXY]).astype(np.float32)

# ------------- motion.npz + CSV + replay script -------------
MOTION_NPZ_PATH = MUJOCO_DIR / "motion.npz"
npz_payload = {
    "qpos": QPOS,
    "fps": np.array([FPS], dtype=np.float32),
    "timestamp_sec": TIMESTAMPS.astype(np.float32),
    "joint_layout": np.array(json.dumps(JOINT_LAYOUT), dtype=object),
}
if OBJ_MJ is not None:
    npz_payload["object_pos"] = OBJ_MJ
np.savez_compressed(MOTION_NPZ_PATH, **npz_payload)

qpos_df = pd.DataFrame(QPOS, columns=[f"qpos_{i}" for i in range(NQ)])
qpos_df.insert(0, "timestamp_sec", TIMESTAMPS)
qpos_df.insert(0, "frame", range(NUM_FRAMES))
# adapters/ is the one place inside the dataset an adapter may write: it holds
# derived, engine-specific views, never canonical data.
(DATASET_DIR / "adapters").mkdir(parents=True, exist_ok=True)
qpos_df.to_csv(DATASET_DIR / "adapters/mujoco_qpos.csv", index=False)

obj_rows = []
for fi in range(NUM_FRAMES):
    if OBJ_MJ is None:
        break
    obj_rows.append([fi, float(TIMESTAMPS[fi]), *np.round(OBJ_MJ[fi], 4),
                     OBJ_SOURCE[fi]])
obj_traj_df = pd.DataFrame(obj_rows, columns=[
    "frame", "timestamp_sec", "x", "y", "z", "position_source"])
obj_traj_df.to_csv(MUJOCO_DIR / "object_trajectory.csv", index=False)
obj_traj_df.to_csv(DATASET_DIR / "adapters/mujoco_object_trajectory.csv", index=False)

REPLAY_SCRIPT = r"""'''Replay Common Behavior motion in the local MuJoCo viewer.

Usage:
    python replay_mujoco.py          # loop playback
    python replay_mujoco.py --once   # play once and hold the last frame

Keys (press inside the viewer window):
    SPACE : pause / resume
    R     : restart from frame 0
    , / . : step one frame backward / forward (while paused)
'''
import sys
import time
from pathlib import Path

import mujoco
import mujoco.viewer
import numpy as np

HERE = Path(__file__).parent
# Load the XML as a string. MjModel.from_xml_path opens the file down in C++
# and fails on non-ASCII paths, which is a common surprise on Windows.
model = mujoco.MjModel.from_xml_string(
    (HERE / "humanoid.xml").read_text(encoding="utf-8"))
data = mujoco.MjData(model)

archive = np.load(HERE / "motion.npz")
qpos = archive["qpos"]
fps = float(archive["fps"][0])
object_pos = archive["object_pos"] if "object_pos" in archive else None
total = len(qpos)
loop = "--once" not in sys.argv

print(f"frames={total}  fps={fps:.2f}  duration={total / fps:.2f}s  "
      f"loop={loop}")
print("keys: SPACE=pause/resume  R=restart  ,/.=step  (progress printed below)")

paused = False
frame = 0


def key_callback(keycode):
    '''Drive playback from the viewer keys (SPACE / R / , / .).'''
    global paused, frame
    key = chr(keycode) if 32 <= keycode < 127 else ""
    if keycode == 32:                     # SPACE
        paused = not paused
    elif key in ("r", "R"):
        frame = 0
    elif key == "." and paused:
        frame = min(frame + 1, total - 1)
    elif key == "," and paused:
        frame = max(frame - 1, 0)


with mujoco.viewer.launch_passive(model, data,
                                  key_callback=key_callback) as viewer:
    while viewer.is_running():
        step_start = time.time()
        index = min(frame, total - 1)
        data.qpos[:] = qpos[index]
        if object_pos is not None and model.nmocap > 0:
            data.mocap_pos[0] = object_pos[index]
        mujoco.mj_forward(model, data)
        viewer.sync()

        # Overwrite one progress line, so first/last frames are easy to spot
        marker = " <<< FIRST" if index == 0 else (
            " >>> LAST" if index == total - 1 else "")
        print(f"\rframe {index + 1:4d}/{total}  "
              f"t={index / fps:6.2f}s{'  [PAUSED]' if paused else ''}"
              f"{marker}          ", end="", flush=True)

        if not paused:
            if frame >= total - 1:
                if loop:
                    print("\n--- loop ---")
                    frame = 0
                else:
                    paused = True         # --once holds on the last frame
            else:
                frame += 1
        wait = 1.0 / fps - (time.time() - step_start)
        if wait > 0:
            time.sleep(wait)
print()
"""
(MUJOCO_DIR / "replay_mujoco.py").write_text(REPLAY_SCRIPT, encoding="utf-8")

(MUJOCO_DIR / "README_MUJOCO.md").write_text("""# MuJoCo Output

Model and motion are kept in separate files:

- `humanoid.xml` ... body / joint definition (MJCF)
- `motion.npz` .... qpos trajectory derived from Common Behavior Data
- `object_trajectory.csv` ... object proxy positions with `position_source`
- `replay_mujoco.py` ... viewer playback (kinematic replay, mj_forward only)
- `mujoco_simulation.mp4` ... rendered video (same trajectory as the viewer)

## Local playback (Windows)

```
pip install mujoco
py -m mujoco.viewer --mjcf="humanoid.xml"   # model only
py replay_mujoco.py                          # model + motion
```

Mode: Kinematic Replay (`data.qpos[:] = recorded_qpos[frame]` + `mj_forward`).
Physically correct contact/forces are out of scope for Demo 2.0.
""", encoding="utf-8")

print("Wrote:", MOTION_NPZ_PATH, "/ replay_mujoco.py / README_MUJOCO.md")


In [ ]:
# @title [A5] Render mujoco_simulation.mp4
# =====================================================================
# MuJoCo adapter 5/6.
# If MuJoCo cannot create a GL context it calls mju_error and aborts the whole
# process -- which in Colab kills the kernel. Rendering in a subprocess lets us
# fall back from egl to osmesa without losing the notebook state.
# =====================================================================
RENDER_WORKER_PATH = WORK_DIR / "mujoco_render_worker.py"
RENDER_CONFIG_PATH = WORK_DIR / "mujoco_render_config.json"
RAW_MUJOCO_PATH = WORK_DIR / "_mujoco_raw.mp4"

caption_meta = [{"frame": fi,
                 "time": f"{TIMESTAMPS[fi]:05.2f}",
                 "action": PHASE[fi]["action"],
                 "phase": PHASE[fi]["phase"],
                 "hand": PHASE[fi]["hand"]} for fi in range(NUM_FRAMES)]

RENDER_WORKER_SOURCE = r'''
import json, os, sys
config = json.loads(open(sys.argv[1], encoding="utf-8").read())
os.environ["MUJOCO_GL"] = config["mujoco_gl"]
if config["mujoco_gl"] == "osmesa":
    os.environ["PYOPENGL_PLATFORM"] = "osmesa"

import cv2
import numpy as np
import mujoco

model = mujoco.MjModel.from_xml_path(config["xml_path"])
data = mujoco.MjData(model)
archive = np.load(config["npz_path"], allow_pickle=True)
qpos = archive["qpos"]
object_pos = archive["object_pos"] if "object_pos" in archive else None

camera = mujoco.MjvCamera()
mujoco.mjv_defaultCamera(camera)
camera.azimuth = 90.0
camera.elevation = -10.0
camera.distance = config["camera_distance"]
camera.lookat[:] = config["camera_lookat"]

width, height = config["width"], config["height"]
print("creating renderer...", flush=True)
renderer = mujoco.Renderer(model, height=height, width=width)
print("renderer ready", flush=True)

writer = cv2.VideoWriter(config["out_path"],
                         cv2.VideoWriter_fourcc(*"mp4v"),
                         config["fps"], (width, height))
captions = config["captions"]
obj_label = config["object_label"]

def put(img, text, x, y, scale=0.55, color=(235, 235, 235)):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale,
                (20, 20, 20), 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale,
                color, 1, cv2.LINE_AA)

for fi in range(len(qpos)):
    data.qpos[:] = qpos[fi]
    if object_pos is not None and model.nmocap > 0:
        data.mocap_pos[0] = object_pos[fi]
    mujoco.mj_forward(model, data)
    # Let the camera drift along with the pelvis
    camera.lookat[0] = float(data.qpos[0])
    renderer.update_scene(data, camera)
    frame = cv2.cvtColor(renderer.render(), cv2.COLOR_RGB2BGR)
    meta = captions[fi]
    put(frame, "MuJoCo Physics Simulation", 12, 26, 0.62)
    put(frame, "Source: Common Behavior Data   Mode: Kinematic Replay", 12, 48, 0.45)
    line3 = "Action: " + meta["action"] + "   Phase: " + meta["phase"]
    if obj_label:
        line3 += "   Object: " + obj_label
    put(frame, line3, 12, 70, 0.45)
    put(frame, "Frame: " + str(meta["frame"]) + "   Time: " + meta["time"] + " s",
        12, height - 14, 0.45)
    writer.write(frame)
    if fi % 60 == 0:
        print("frame", fi, "/", len(qpos), flush=True)

writer.release()
renderer.close()
print("done", flush=True)
'''
RENDER_WORKER_PATH.write_text(RENDER_WORKER_SOURCE, encoding="utf-8")

render_config = {
    "xml_path": str(HUMANOID_XML_PATH),
    "npz_path": str(MOTION_NPZ_PATH),
    "out_path": str(RAW_MUJOCO_PATH),
    "width": MUJOCO_CFG["render_width"],
    "height": MUJOCO_CFG["render_height"],
    "fps": FPS,
    "camera_distance": MUJOCO_CFG["camera_distance"],
    "camera_lookat": [0.0, 0.0, 0.90],
    "captions": caption_meta,
    "object_label": PRIMARY_LABEL,
    "mujoco_gl": None,
}

MUJOCO_MP4 = MUJOCO_DIR / "mujoco_simulation.mp4"
render_success = False
for gl_backend in (["egl", "osmesa"] if HAS_GPU else ["osmesa"]):
    render_config["mujoco_gl"] = gl_backend
    RENDER_CONFIG_PATH.write_text(json.dumps(render_config), encoding="utf-8")
    print(f"--- Rendering with MuJoCo (backend={gl_backend}, subprocess) ---")
    result = subprocess.run(
        [sys.executable, str(RENDER_WORKER_PATH), str(RENDER_CONFIG_PATH)],
        capture_output=True, text=True)
    if result.returncode == 0 and RAW_MUJOCO_PATH.exists():
        render_success = True
        break
    print(f"backend={gl_backend} failed (returncode={result.returncode})")
    print("--- stderr (tail) ---")
    print("\n".join(result.stderr.splitlines()[-8:]))

if render_success:
    encode_h264(RAW_MUJOCO_PATH, MUJOCO_MP4, FPS)
    print("Wrote:", MUJOCO_MP4)
    show_video(MUJOCO_MP4)
else:
    print("MuJoCo rendering failed.")
    print("Try: switch to a GPU runtime and restart, or lower "
          "MUJOCO_CFG['render_width'] to 640 in [A3] and re-run [A3]-[A5].")


In [ ]:
# @title [A6] Adapter report + acceptance check + package
# =====================================================================
# MuJoCo adapter 6/6.
# =====================================================================
def print_checks(title, checks):
    print(f"=== Acceptance criteria ({title}) ===")
    for label, ok in checks:
        print(("  [x] " if ok else "  [ ] ") + label)


def package_adapter_output(zip_name, entries):
    """Zip this adapter's output with paths relative to the project root.

    `entries` may hold directories or single files. Unzipping the result into
    /content/human_behavior_demo_2_0/ on another runtime puts every file back
    where the other notebooks expect it.
    """
    zip_path = PROJECT_DIR / zip_name
    members = []
    for entry in entries:
        entry = Path(entry)
        if entry.is_dir():
            members += [p for p in sorted(entry.rglob("*")) if p.is_file()]
        elif entry.is_file():
            members.append(entry)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in members:
            archive.write(path, path.relative_to(PROJECT_DIR))
    print(f"\nAdapter output: {zip_path} "
          f"({zip_path.stat().st_size / 1024 / 1024:.1f} MB, "
          f"{len(members)} files)")
    print(f"  colab download -s <session> {zip_path} ./{zip_name}")
    if SHOW_MEDIA:
        try:
            from google.colab import files as colab_files
            if zip_path.stat().st_size < 200 * 1024 * 1024:
                colab_files.download(str(zip_path))
            else:
                print("  (over 200 MB: use the file pane on the left)")
        except Exception:  # noqa: BLE001
            print("  (automatic download unavailable; use the file pane)")
    return zip_path
REPORT = write_adapter_report(
    MUJOCO_DIR / "adapter_report.json",
    adapter="mujoco",
    target="MJCF humanoid + qpos trajectory (kinematic replay)",
    reads=["manifest.json", "behavior_summary.json", "timeline/frames.jsonl"],
    unrepresented=[
        "face landmarks and blendshapes: the MJCF humanoid has no face rig",
        "per-joint finger flexion: each finger is a single hinge driven by the "
        "mean curl, so the three joint angles collapse into one",
        "hand orientation detail beyond the wrist ball joint",
        "physical contact and forces: playback is kinematic replay "
        "(qpos assignment + mj_forward), gravity is disabled",
        "object rotation: the proxy body is positioned, never oriented",
        "captions: written to the video's caption band only, not into the model",
    ])

CHECKS = [
    ("Input: canonical dataset located",
     (DATASET_DIR / "manifest.json").exists()),
    ("Model and motion kept in separate files",
     HUMANOID_XML_PATH.exists() and MOTION_NPZ_PATH.exists()),
    ("Model: humanoid.xml compiles", int(mj_model.nq) > 0),
    ("Motion: qpos covers every frame", QPOS.shape[0] == NUM_FRAMES),
    ("Motion: qpos width matches the compiled model", QPOS.shape[1] == NQ),
    ("Object: trajectory exported with position_source",
     (MUJOCO_DIR / "object_trajectory.csv").exists()),
    ("Local playback: replay_mujoco.py",
     (MUJOCO_DIR / "replay_mujoco.py").exists()),
    ("Render: mujoco_simulation.mp4", MUJOCO_MP4.exists()),
    ("Dataset view: adapters/mujoco_qpos.csv",
     (DATASET_DIR / "adapters/mujoco_qpos.csv").exists()),
    ("Output: adapter_report.json written",
     (MUJOCO_DIR / "adapter_report.json").exists()),
]
print_checks("mujoco adapter", CHECKS)

package_adapter_output("mujoco_adapter_output.zip", [
    MUJOCO_DIR,
    DATASET_DIR / "adapters/mujoco_qpos.csv",
    DATASET_DIR / "adapters/mujoco_object_trajectory.csv",
])
print("\nLocal playback: unzip, then `pip install mujoco && python "
      "replay_mujoco.py` inside output/02_mujoco/.")
print("Next: cbd_adapter_mediapipe_overlay.ipynb and cbd_adapter_unity_vrm.ipynb "
      "read the same dataset. A three-screen comparison of all three ships in "
      "examples/human-capture/sample_output/05_comparison/.")
